<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/09-serving-inference/02-inference-performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inference Performance (concept)

**Goal:** Understand the levers behind an LLM server's throughput and latency (continuous batching, the KV cache, quantization, and the throughput-vs-latency trade), and do the napkin math to size a deployment.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## Why this notebook exists

[Notebook 01](01-serving-frameworks.ipynb) said a serving framework's whole job is to answer *many* requests on *expensive* GPUs efficiently, and that this one constraint drives every optimization. This notebook is those optimizations. It's the difference between "I used vLLM" and being able to answer *why vLLM is fast* and *how many GPUs your service needs*, the questions an AI-systems interview actually presses on.

Like 01, it's **concept-first**: the mental models and the sizing math run on any runtime (CPU is fine, it's arithmetic). An optional fenced appendix at the end measures a real batching curve on a GPU.

> **⭐ Key takeaway —** LLM inference is **memory-bandwidth-bound, not compute-bound**. The GPU spends most of its time *moving weights and cache*, not doing math. Almost every performance lever below is really a trick to move less memory, or to reuse a GPU that's otherwise sitting idle waiting on memory.

## First, the two numbers everyone conflates

Performance is never one number. Two, and they trade off:

- **Latency:** how long *one* request takes. Split further into **TTFT** (time to first token, the *prefill* of your prompt) and **TPOT** (time per output token, each *decode* step). A chat UI lives or dies on TTFT; a batch job only cares about total.
- **Throughput:** how many tokens/requests the server does *per second across everyone*. This is what sets your GPU bill.

They fight: the main throughput lever (batching, below) *raises* per-request latency. "Fast" is meaningless until you say **for whom**: the single user waiting, or the fleet-wide token bill.

> **🔵 Interview signal —** when asked to "make inference faster," the strong move is to ask back: *"latency or throughput, and is this interactive or batch?"* Naming the trade-off before optimizing is the senior tell.

## Lever 1 — Continuous batching (the big one)

A GPU running one request at a time is almost entirely idle: one decode step barely touches its compute while it waits on memory. **Batching** runs many requests' decode steps *together*, so one expensive weight-read serves the whole batch. That's the throughput multiplier.

Naive ("static") batching waits for the whole batch to finish before starting the next, so one long generation stalls everyone. **Continuous batching** (vLLM's headline feature) instead swaps finished requests out and new ones in *every step*, keeping the GPU packed. Same hardware, several times the throughput on real traffic.

The cell below is a simplified model of *why* it wins: no GPU, just the idle-time arithmetic.

In [ ]:
# Toy model: a GPU step costs the same whether it serves 1 request or N (up to a cap),
# because the cost is dominated by reading the weights once. So batching amortizes that read.
STEP_MS = 20          # one decode step: ~fixed, memory-bound, regardless of batch size (until saturation)
TOKENS_PER_REQ = 200  # output tokens per request

def throughput(batch_size, step_ms=STEP_MS):
    # N requests decode in lockstep: N*tokens produced over tokens*step_ms wall time.
    reqs = batch_size
    wall_s = (TOKENS_PER_REQ * step_ms) / 1000
    return reqs / wall_s, (TOKENS_PER_REQ * batch_size) / wall_s  # req/s, tok/s

print(f"{'batch':>5} {'req/s':>8} {'tok/s':>9}   note")
for b in [1, 4, 16, 32]:
    r, t = throughput(b)
    note = "GPU mostly idle" if b == 1 else "amortizing the weight read across the batch"
    print(f"{b:>5} {r:>8.1f} {t:>9.0f}   {note}")
print("\n-> same GPU, same per-step cost: throughput scales with batch until memory (the KV cache) runs out.")

> **⚠️ Production reality —** batching is why your *own* latency can *rise* under load even though the server is "faster." Your request now shares each GPU step with others. That's the throughput/latency trade made concrete, and why interactive services cap batch size or reserve headroom, while batch pipelines crank it up.

## Lever 2 — The KV cache (what caps the batch)

When the model generates token 50, it needs the keys/values it already computed for tokens 1–49. Recomputing them every step would be quadratic; instead the server **caches** them in GPU memory, the **KV cache**. It grows with *every* token, for *every* request in the batch.

That makes the KV cache, not compute, the usual ceiling on batch size:

- It's why a GPU with plenty of compute still can't batch more: it's **out of KV-cache memory**, not out of math.
- It's what **vLLM's PagedAttention** optimizes, managing the cache in fixed pages like OS virtual memory, so fragmentation doesn't waste it. That's the mechanism behind 01's "far more concurrent requests per GPU."
- It's why **long contexts are expensive**: cache size is proportional to sequence length. Doubling context roughly doubles cache per request, roughly halving how many you can batch.

The cell sizes a KV cache so the ceiling is a number, not a vibe.

In [ ]:
# KV-cache size = 2 (K and V) * layers * seq_len * hidden * bytes_per_val.
# Numbers below are ~a 7B model in fp16. This is the calculation that sets your batch ceiling.
def kv_cache_gb(seq_len, layers=32, hidden=4096, bytes_per_val=2):
    bytes_ = 2 * layers * seq_len * hidden * bytes_per_val
    return bytes_ / 1e9

GPU_KV_BUDGET_GB = 12  # e.g. a 24GB GPU, ~half left for the cache after weights

print(f"{'context':>8} {'KV/req':>9} {'max batch':>10}")
for ctx in [512, 2048, 8192, 32768]:
    per = kv_cache_gb(ctx)
    print(f"{ctx:>8} {per:>7.2f}GB {int(GPU_KV_BUDGET_GB/per):>9}")
print(f"\n-> with ~{GPU_KV_BUDGET_GB}GB for cache: long contexts collapse the batch, so throughput drops. "
      "Context length is a capacity decision, not just a quality one.")

## Lever 3 — Quantization (move less memory)

Since inference is memory-bandwidth-bound, storing weights (and cache) in **fewer bits** moves less memory, so it's both *faster* and *smaller*. Serving in 8-bit (INT8/FP8) or 4-bit (e.g. AWQ, GPTQ) roughly halves or quarters the memory footprint and bandwidth vs fp16:

- **Fits a bigger model on a smaller GPU:** a 13B model in 4-bit fits where fp16 wouldn't.
- **Frees memory for the KV cache** → bigger batches → more throughput (compounds with levers 1–2).
- **Costs some quality**, usually small at 8-bit, more noticeable at 4-bit. This is where your **section-04 eval harness** earns its keep: quantization is a change, so you *measure* it, you don't eyeball it.

> **💡 Why it's nearly free lunch —** because the bottleneck is memory movement, not arithmetic precision, dropping bits often barely dents quality while directly buying speed and capacity. It's usually the first knob after batching.

## Putting it together — sizing a deployment

The interview question is rarely "explain PagedAttention." It's *"you expect 50 requests/second, how many GPUs?"* That's arithmetic built from the levers above. Do it cold:

In [ ]:
def gpus_needed(target_req_per_s, tokens_per_req, tok_per_s_per_gpu, util=0.7):
    """Size a fleet. tok_per_s_per_gpu is your measured batched throughput (lever 1);
    util<1 leaves headroom so p99 latency doesn't explode (lever 1's trade-off)."""
    demand_tok_s = target_req_per_s * tokens_per_req
    effective = tok_per_s_per_gpu * util
    import math
    return demand_tok_s, math.ceil(demand_tok_s / effective)

# Scenario: 50 req/s, 300 tok each, a GPU measured at ~2500 batched tok/s.
demand, n = gpus_needed(target_req_per_s=50, tokens_per_req=300, tok_per_s_per_gpu=2500)
print(f"demand: {demand:.0f} tok/s")
print(f"GPUs needed (70% util for headroom): {n}")
print("\nlevers that change this number:")
print("  - bigger batch / PagedAttention -> higher tok/s/gpu -> fewer GPUs")
print("  - quantization -> higher tok/s/gpu AND cheaper GPUs")
print("  - longer contexts -> smaller batch -> lower tok/s/gpu -> MORE GPUs")

> **🔵 Interview signal —** carry these four moves into the room: (1) restate as *latency or throughput*, (2) *demand = req/s × tokens/req*, (3) *GPUs = demand / (throughput/GPU × utilization)*, (4) name the lever that moves throughput/GPU (batching, quantization, context length). That structure *is* the ML-system-design answer, worked fully in [section 10](../10-ml-system-design/01-designing-an-inference-service.ipynb).

## Practices & anti-patterns

| ✅ Do | ❌ Anti-pattern |
|---|---|
| State latency **or** throughput (and interactive vs batch) before optimizing | "Make it faster" with no metric, then optimize the wrong one |
| Split latency into TTFT (prefill) and TPOT (decode) | Report one "latency" number and hide the TTFT the user actually feels |
| Treat the **KV cache**, not compute, as your batch ceiling | Buy a bigger-compute GPU when you were out of cache memory |
| Size context length as a *capacity* decision (it shrinks the batch) | Set max context to a huge value "to be safe" and wonder why throughput tanked |
| Measure quantized quality with the section-04 evals before shipping | Ship 4-bit because it's faster; discover the quality regression in prod |
| Size fleets with demand ÷ (throughput/GPU × utilization), leave headroom | Run GPUs at 100% util, then watch p99 latency blow up under a spike |

## Optional appendix — measure a real batching curve (needs a GPU)

> **⚠️ This appendix needs a GPU runtime and is skippable.** In Colab: **Runtime → Change runtime type → T4 GPU**. On CPU, read it and move on; the math above is the graded material.

This appendix is **self-contained**: each Colab notebook is its own runtime, so it stands up its *own* vLLM server (you can't reuse one from notebook 01). It's the same launch as [01's appendix](01-serving-frameworks.ipynb), then it sends the same prompt at rising concurrency so you can *see* lever 1: tokens/sec climbs with the batch, then flattens as the KV cache saturates.

In [ ]:
# Appendix cell 1 — GPU check. Stop here if this prints "no GPU".
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(gpu.stdout.strip() or "no GPU — the batching math above is what's graded; skip this appendix")

In [ ]:
# Appendix cell 2 — install vLLM + fix the torchaudio CUDA-version clash (see notebook 01 for why).
# vLLM upgrades PyTorch; Colab's stale torchaudio then fails to import and would crash the server.
%pip install -q vllm
!pip uninstall -y torchaudio 2>/dev/null || true
!python -c "import transformers, vllm; print('imports OK — vllm', vllm.__version__)"
# On a fresh runtime no restart is needed. If the line above errors (pip upgraded an already-loaded
# package), Runtime -> Restart session, then rerun from the GPU-check cell SKIPPING this %pip line.

In [ ]:
# Appendix cell 3 — launch this notebook's own vLLM server (background, logs to a file).
#   --enforce-eager: faster/safer T4 startup   --gpu-memory-utilization 0.85: leave headroom
import subprocess, time, urllib.request

LOG = "/content/vllm.log"
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
     "--max-model-len", "2048", "--port", "8000",
     "--enforce-eager", "--gpu-memory-utilization", "0.85"],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

up = False
for _ in range(90):                       # up to ~7.5 min (model download + load)
    if server.poll() is not None:
        print(f"server exited (code {server.returncode}). Log tail:\n")
        print("".join(open(LOG).readlines()[-25:])); break
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2)
        print("vLLM server is up on :8000"); up = True; break
    except Exception:
        time.sleep(5)
if not up and server.poll() is None:
    print("still starting. Log tail:\n"); print("".join(open(LOG).readlines()[-15:]))

In [ ]:
# Appendix cell 4 — the batching curve. Fire N concurrent requests, measure aggregate tokens/sec.
# NOTE: "server is up" (HTTP layer) can precede the engine's warm-up, so the first request may race
# it — we retry once before measuring, and bail with the log tail if the server actually died.
import time, concurrent.futures
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
PROMPT = "Explain what a KV cache is in two sentences."

def one_call(_=None):
    r = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": PROMPT}], max_tokens=128)
    return r.usage.completion_tokens

# Warm up / wait for the engine to actually accept requests (handles the startup race).
ready = False
for attempt in range(1, 13):
    if server.poll() is not None:
        print(f"server process exited (code {server.returncode}). Log tail:\n")
        print("".join(open("/content/vllm.log").readlines()[-25:])); break
    try:
        one_call(); ready = True; break
    except Exception as e:
        print(f"warm-up attempt {attempt}: not ready ({type(e).__name__}) — retrying in 5s"); time.sleep(5)

if ready:
    print(f"\n{'concurrency':>11} {'tok/s':>8}")
    for c in [1, 2, 4, 8, 16]:
        start = time.time()
        with concurrent.futures.ThreadPoolExecutor(max_workers=c) as ex:
            toks = sum(ex.map(one_call, range(c)))
        dt = time.time() - start
        print(f"{c:>11} {toks/dt:>8.0f}")
    print("\n-> tok/s should rise with concurrency (continuous batching), then flatten as the KV cache fills.")
else:
    print("\nserver never became ready — rerun cell 3, then this cell.")

In [ ]:
# Appendix cell 5 — free the GPU. Only one vLLM can own the card; a leftover process causes the
# "free memory" error next time. (Freeing VRAM != releasing the GPU: a connected runtime still
# burns your Colab GPU quota — use Runtime -> Disconnect and delete runtime when truly done.)
server.terminate()
import time; time.sleep(3)
!pkill -9 -f vllm.entrypoints 2>/dev/null || true
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

## Exercises

1. **Size your capstone.** Pick a plausible load for the section-12 capstone (req/s, tokens/req) and a throughput/GPU number (measure it in the appendix, or assume ~2500 tok/s). Compute the GPU count. Then re-run with 4× the context length and explain, in one sentence, why the number jumped.
2. **Latency or throughput?** For three products (a live chat assistant, a nightly document-summarization job, and a code-completion IDE plugin) say which metric dominates and one serving choice it implies (batch size, model size, or quantization).
3. **The cache is the ceiling.** Using the `kv_cache_gb` cell, find the max context length that still lets you batch **8** requests in a 12 GB cache budget. Show the number and say what you'd trade to double it.
4. **Quantify quantization.** Describe (no code) how you'd use the section-04 eval harness to decide whether 4-bit serving is acceptable for your capstone: what you'd measure, against what baseline, and the threshold that would make you keep fp16.